# SOLikeT grid yaml generator

This notebook writes the cobaya-grid input of the ISO analysis and creates the grid, which can then
be run with:
```
cobaya-grid-run <grid_folder>
```

It holds no configuration of its own. The theory code, data level, models, datasets and groups are
listed in the configuration's `grid_spec.yaml`, and every piece of the runs comes from the private
configuration (`$ISO_CONFIG`) through `iso_config.py`, the same functions the other notebooks use.
The grid yaml and folder hold machine paths and private configuration, so they are written under
`$ISO_DATA/runs/`. Section 3 checks that each grid job runs exactly the `info` that
`iso_config.build_info(dataset, model, level=LEVEL)` gives in a notebook.

## 1 — Configuration

In [ ]:
import iso_config

spec = iso_config.grid_spec()
LEVEL = spec["level"]  # or "mock", "twin", "real" to override grid_spec.yaml

OUTPUT_YAML = iso_config.runs_dir() / f"grid_{LEVEL}.yaml"
OUTPUT_DIR = iso_config.runs_dir() / f"grid_{LEVEL}"

print("theory code:", spec["theory"], "| level:", LEVEL)
for name, group in spec["groups"].items():
    print(f"group {name}: models {group['models']} x datasets {group['datasets']}")

## 2 — Write the grid yaml and create the grid

In [ ]:
import yaml
from cobaya.grid_tools.gridconfig import grid_create

grid = iso_config.grid_info(LEVEL)

header = (
    "# SOLikeT grid yaml -- generated by generate_grid.ipynb from the ISO configuration.\n"
    "# Do not edit by hand (nor commit); edit the configuration and re-run the notebook.\n"
    "#   cobaya-grid-create <grid_folder> <this file>\n\n"
)
text = yaml.safe_dump(grid, sort_keys=False, default_flow_style=False, allow_unicode=True)
OUTPUT_YAML.parent.mkdir(parents=True, exist_ok=True)
OUTPUT_YAML.write_text(header + text)
print(f"Written: {OUTPUT_YAML}  ({OUTPUT_YAML.stat().st_size} bytes)")

grid_create((str(OUTPUT_DIR), str(OUTPUT_YAML)))

## 3 — Check: grid jobs = notebook configs

`cobaya-grid` writes one input file per (model, dataset) job. Each must be identical to
`iso_config.build_info(dataset, model, level=LEVEL)`, apart from the `output` entry the grid adds.

In [ ]:
from cobaya.yaml import yaml_load_file

theory = spec["theory"]
n_jobs = 0
for group in spec["groups"].values():
    for model in group["models"]:
        for dataset in group["datasets"]:
            job_file = OUTPUT_DIR / "input_files" / f"{theory}_{model}_{dataset}.yaml"
            job = yaml_load_file(str(job_file))
            job.pop("output")
            expected = iso_config.build_info(dataset, model, level=LEVEL)
            assert job == expected, f"{job_file.name} differs"
            n_jobs += 1
print(f"all {n_jobs} grid jobs match iso_config.build_info")